In [1]:
begin
    using Pkg
    dev_folder = joinpath(@__DIR__, "../Examples")
    Pkg.activate(dev_folder)
end
Threads.nthreads()

  Activating project at `~/Realizibility_index/BindingAndCatalysis.jl/Examples`


24

In [2]:
using Polyhedra
using GLMakie # for plotting, we use Makie backend, could also be GLMakie or WGLMakie...
using Revise
using BindingAndCatalysis # import the package

[ Info: Precompiling BindingAndCatalysis [b532a4b1-2c45-4c12-bcaf-8695372aa40c] (cache misses: include_dependency fsize change (2), incompatible header (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :

In [6]:
using SparseArrays
"""
Canonical hyperplane

Stored in canonical form with `u < v`:

    z_u - z_v + log(num/den) = 0

where `(num, den)` is the reduced integer ratio.
"""
struct Hyperplane_perm
    u::Int # fast access 
    v::Int # fast access
    num::Int # reduced positive integer
    den::Int # reduced positive integer
    c0::Float64 # pre-logarithm 
    crow::SparseVector{Int8,Int}      # +1 at u, -1 at v
    crow_neg::SparseVector{Int8,Int}  # +1 at v, -1 at u
end

"""
Helper struct for managing matrix operations.
- `J[i]`: positive columns in row i
- `choice_slot[i][p]`: local slot of column p inside J[i], or 0 if p ∉ J[i]
- `choice_map[i][t]`: all oriented inequalities for choosing p = J[i][t]
- `hyperplanes`: global deduplicated hyperplane pool
- `asymptotic`: all asymptotic regimes
- `feasible`: all regimes feasible under the weighted constraints
"""
struct Matrix_helper
    n::Int
    J::Vector{Vector{Int}}
    choice_slot::Vector{Vector{Int}}
    choice_logcoeff::Vector{Vector{Float64}}
    rowptr::Vector{Int}
    total_constraints::Int
    choice_map::Dict{Tuple{Int,Int},Int}
    hyperplanes::Vector{Hyperplane_perm}
end


function _build_matrix_helper(L::AbstractMatrix{Tv}) where {Tv<:Integer}
    d, n = size(L)
    J = Vector{Vector{Int}}(undef, d)
    choice_slot = [zeros(Int, n) for _ in 1:d]
    choice_logcoeff = Vector{Vector{Float64}}(undef, d)
    choice_map = Vector{Vector{Vector{ChoiceIneq}}}(undef, d)

    # Global deduplicated hyperplane pool
    key_to_id = Dict{Tuple{Int,Int,Tv,Tv}, Int}()
    hyperplanes = Hyperplane_perm{Tv}[]

    # First build J / choice_map
    @inbounds for i in 1:d
        Ji = Int[]
        sizehint!(Ji, n)
        for j in 1:n
            if L[i, j] > 0
                push!(Ji, j)
            end
        end
        isempty(Ji) && throw(ArgumentError("row $i of L has no positive entry"))

        J[i] = Ji
        choice_logcoeff[i] = [log(Float64(L[i, j])) for j in Ji]

        row_choices = Vector{Vector{ChoiceIneq}}(undef, length(Ji))

        for (t, p) in pairs(Ji)
            choice_slot[i][p] = t

            refs = Vector{ChoiceIneq}(undef, max(length(Ji) - 1, 0))
            ptr = 1
            Lp = L[i, p]

            for k in Ji
                k == p && continue
                Lk = L[i, k]

                # Canonicalize the hyperplane by ordering the variable pair.
                if p < k
                    u, v = p, k
                    num, den = _reduced_ratio(Lp, Lk)
                    sign = Int8(+1)
                else
                    u, v = k, p
                    num, den = _reduced_ratio(Lk, Lp)
                    sign = Int8(-1)
                end

                key = (u, v, num, den)
                hid = get(key_to_id, key, 0)

                if hid == 0
                    crow = sparsevec([u, v], Int8[1, -1], n)
                    crow_neg = sparsevec([v, u], Int8[1, -1], n)
                    c0 = log(Float64(num)) - log(Float64(den))
                    push!(hyperplanes, Hyperplane_perm{Tv}(u, v, num, den, c0, crow, crow_neg))
                    hid = length(hyperplanes)
                    key_to_id[key] = hid
                end

                refs[ptr] = ChoiceIneq(
                    hid,
                    sign,
                    k,
                    log(Float64(Lp)) - log(Float64(Lk))
                )
                ptr += 1
            end

            row_choices[t] = refs
        end

        choice_map[i] = row_choices
    end

    # Row block pointers for regime constraints:
    # row i contributes |J_i|-1 inequalities, independent of the chosen dominant.
    rowptr = Vector{Int}(undef, d + 1)
    rowptr[1] = 1
    @inbounds for i in 1:d
        rowptr[i + 1] = rowptr[i] + (length(J[i]) - 1)
    end
    total_constraints = rowptr[end] - 1
    
    return Matrix_helper(
        n, J, choice_slot, choice_logcoeff, rowptr, total_constraints,choice_map,hyperplanes
    )
end

_build_matrix_helper (generic function with 1 method)

In [ ]:
a = _build_matrix_helper()

In [10]:
P = [1 -1]

pi = [0 0 1;
       0 1 0]
H =[-1;1;0]

3-element Vector{Int64}:
 -1
  1
  0

In [11]:
P* pi *H

1-element Vector{Int64}:
 -1

In [12]:
model = let 
    L = [0 1 1; 1 0 1]
    N = [1 1 -1]
    Bnc(L=L,N=N)
end

----------Binding Network Summary:-------------
Number of species (n): 3
Number of conserved quantities (d): 2
Number of reactions (r): 1
L matrix: [0 1 1; 1 0 1]
N matrix: [1 1 -1]
Direction of binding reactions: forward
Catalysis involved: No
Regimes constructed: No
-----------------------------------------------

In [19]:
get_H(model,2)

3×3 SparseMatrixCSC{Float64, Int64} with 5 stored entries:
 -1.0  1.0  1.0
  1.0   ⋅    ⋅ 
   ⋅   1.0   ⋅ 

In [23]:
M1= get_M(model,2)

3×3 SparseMatrixCSC{Int64, Int64} with 5 stored entries:
 ⋅  1   ⋅
 ⋅  ⋅   1
 1  1  -1

In [24]:
M2 = get_M(model,4)

3×3 SparseMatrixCSC{Int64, Int64} with 5 stored entries:
 ⋅  ⋅   1
 ⋅  ⋅   1
 1  1  -1

In [22]:
a = inv([0 0 1 ;1 1 -1;0 -1 1])

3×3 Matrix{Float64}:
 -0.0  1.0   1.0
  1.0  0.0  -1.0
  1.0  0.0   0.0

In [25]:
M1 *a

3×3 Matrix{Float64}:
 1.0  0.0  -1.0
 1.0  0.0   0.0
 0.0  1.0   0.0

In [26]:
M2 *a

3×3 Matrix{Float64}:
 1.0  0.0  0.0
 1.0  0.0  0.0
 0.0  1.0  0.0